# Chapitre 3 - Cours 4 : Index secondaires, multidimensionnels, plein texte et en mémoire

Ce notebook couvre la section *Other Indexing Structures* : les index secondaires, le
rangement des valeurs dans l'index (*Storing values within the index* : index groupé,
fichier de tas, index couvrant), les index multicolonnes (*Multi-column indexes*), la
recherche plein texte et floue (*Full-text search and fuzzy indexes*) et les bases en
mémoire (*Keeping everything in memory*). Les expériences utilisent nos propres
structures et la bibliothèque standard `sqlite3`.

**Plan.**

1. Les index secondaires : des clés non uniques.
2. Où ranger les données : fichier de tas, index groupé, index couvrant.
3. Les index multicolonnes : index concaténé et index multidimensionnel.
4. La recherche plein texte : index inversé, intersection, recherche floue.
5. Tout garder en mémoire.

In [1]:
import random
import sqlite3
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import storage as S
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)

## 1. Les index secondaires (section *Other Indexing Structures*)

Jusqu'ici, nos index étaient des index **clé-valeur**, comme une clé primaire : la clé
identifie un enregistrement unique (une ligne, un document, un sommet), et les autres
enregistrements s'y réfèrent par cette clé. Il est très courant d'ajouter des **index
secondaires** (`CREATE INDEX` en SQL) pour chercher par d'autres colonnes, par exemple
toutes les commandes d'un client par `user_id`.

**La différence : les valeurs indexées ne sont pas uniques.** Deux façons de s'en
accommoder :

1. chaque entrée de l'index contient la **liste** des identifiants de lignes qui
   correspondent (comme la *postings list* d'un index plein texte, section 4) ;
2. on rend chaque entrée **unique** en lui ajoutant l'identifiant de ligne :
   la clé de l'index devient `(user_id, id_ligne)`. Toutes les lignes d'un même client
   sont alors **voisines** dans l'ordre de l'index : une requête devient un parcours
   d'intervalle.

B-trees et arbres LSM conviennent tous deux. Implémentons la deuxième méthode avec
notre arbre B+.

In [2]:
rng = np.random.default_rng(0)
N = 50_000
commandes = pd.DataFrame({"id": np.arange(N), "user_id": S.zipf_keys(N, 5_000, s=0.8, seed=3),
                          "montant": np.round(rng.lognormal(3, 0.8, N), 2)})
primaire = S.BPlusTree(max_keys=64)
secondaire = S.BPlusTree(max_keys=64)
for row in commandes.itertuples(index=False):
    primaire.insert(int(row.id), (int(row.user_id), float(row.montant)))
    secondaire.insert((int(row.user_id), int(row.id)), None)  # clé unique : (user_id, id)

client = int(commandes.user_id.value_counts().index[0])
secondaire.page_reads = primaire.page_reads = 0
ids = [i for (u, i), _ in secondaire.range((client, -1), (client, N))]
lignes = [primaire.get(i) for i in ids]
print(f"client {client} : {len(ids)} commandes trouvées ;",
      f"pages lues dans l'index secondaire : {secondaire.page_reads},",
      f"dans l'index primaire : {primaire.page_reads} ({primaire.height} par commande)")

client 801 : 2244 commandes trouvées ; pages lues dans l'index secondaire : 73, dans l'index primaire : 6732 (3 par commande)


L'index secondaire trouve les identifiants en un seul parcours d'intervalle (quelques
pages) ; il faut ensuite aller chercher **chaque ligne** via l'index primaire, ce qui
coûte une descente complète par ligne. D'où la question de la section suivante : où
ranger les données elles-mêmes ?

> **À retenir.** Un index secondaire indexe une colonne non unique ; on le construit
> comme un index clé-valeur, soit avec des listes d'identifiants, soit en ajoutant
> l'identifiant à la clé. Chaque correspondance renvoie à la ligne complète, ailleurs.